In [70]:
import numpy as np
import matplotlib.pyplot as plt
from MieSppForce.simulation import SphericalGrid, DiagramCalculator, SimulationConfig, FieldsCalculator
from numpy import pi, sin, cos
from pint import UnitRegistry
ureg = UnitRegistry()

base_config = SimulationConfig(
    wl=900*ureg.nm,
    #     R = 110.5*ureg.nm,
    R=110*ureg.nm,
    dist=2 * ureg.nm,
    angle=np.deg2rad(25),
    psi=np.deg2rad(45),
    chi=np.deg2rad(22.5),
    substrate='Au',
    particle='Si',
    show_warnings=True,
    initial_field_type='plane_wave'
)

r_multiplyer = 1

Grid3D_Reg = SphericalGrid(
    r=base_config.wl*r_multiplyer,
    theta=np.linspace(0, pi/2, 100)*ureg.rad,
    phi=np.linspace(0, 2*pi, 100) * ureg.rad
)

# Grid3D_SPP = SphericalGrid(
#     r=base_config.wl*r_multiplyer,
#     theta=np.linspace(0, pi/2, 100)*ureg.rad,
#     phi=np.linspace(0, 2*pi, 100) * ureg.rad
# )

In [71]:
# diagReg = DiagramCalculator(base_config, Grid3D_Reg, normalize=None).compute('reg')

# diagSPP = DiagramCalculator(base_config, Grid3D_SPP, normalize=None).compute('spp')

diagTot = DiagramCalculator(base_config, Grid3D_Reg, normalize=None).compute()

100%|██████████| 10000/10000 [00:12<00:00, 832.09it/s]


In [16]:
field_reg = FieldsCalculator(base_config).compute(Grid3D_Reg, 'reg')
field_spp = FieldsCalculator(base_config).compute(Grid3D_Reg, 'spp')

100%|██████████| 5000/5000 [00:06<00:00, 804.90it/s]


In [36]:
from MieSppForce.simulation import DiagramResult

def compute_intensity(E, H):
    Ex, Ey, Ez = E
    Hx, Hy, Hz = H

    Sx = 0.5*np.real(Ey*Hz.conj() - Ez*Hy.conj())
    Sy = 0.5*np.real(Ez*Hx.conj() - Ex*Hz.conj())
    Sz = 0.5*np.real(Ex*Hy.conj() - Ey*Hx.conj())

    return np.array([Sx, Sy, Sz])
    
    

Ex_reg = field_reg.df.Ex.apply(lambda x: x.magnitude).to_numpy()
Ey_reg = field_reg.df.Ey.apply(lambda x: x.magnitude).to_numpy()
Ez_reg = field_reg.df.Ez.apply(lambda x: x.magnitude).to_numpy()
Hx_reg = field_reg.df.Hx.apply(lambda x: x.magnitude).to_numpy()
Hy_reg = field_reg.df.Hy.apply(lambda x: x.magnitude).to_numpy()
Hz_reg = field_reg.df.Hz.apply(lambda x: x.magnitude).to_numpy()

Ex_spp = field_spp.df.Ex.apply(lambda x: x.magnitude).to_numpy()
Ey_spp = field_spp.df.Ey.apply(lambda x: x.magnitude).to_numpy()
Ez_spp = field_spp.df.Ez.apply(lambda x: x.magnitude).to_numpy()
Hx_spp = field_spp.df.Hx.apply(lambda x: x.magnitude).to_numpy()
Hy_spp = field_spp.df.Hy.apply(lambda x: x.magnitude).to_numpy()
Hz_spp = field_spp.df.Hz.apply(lambda x: x.magnitude).to_numpy()

E_reg = [Ex_reg, Ey_reg, Ez_reg]
E_spp = [Ex_spp, Ey_spp, Ez_spp]
H_reg = [Hx_reg, Hy_reg, Hz_reg]
H_spp = [Hx_spp, Hy_spp, Hz_spp]


r = field_reg.df.r.apply(lambda x: x.magnitude).to_numpy()
phi = field_reg.df.phi.apply(lambda x: x.magnitude).to_numpy()
z = field_reg.df.z.apply(lambda x: x.magnitude).to_numpy()

S_mix1 = compute_intensity(E_spp, H_reg)
S_mix2 = compute_intensity(E_reg, H_spp)

theta = np.arctan2(r, z)
Sx = S_mix1[0] + S_mix2[0]
Sy = S_mix1[1] + S_mix2[1]
Sz = S_mix1[2] + S_mix2[2]

I  = Sx * np.sin(theta) * np.cos(phi) + Sy * np.sin(theta) * np.sin(phi) + Sz * np.cos(theta)



Diag_mix = DiagramResult(phi, theta, I)


In [20]:
import pandas as pd
from MieSppForce.simulation import DiagramResult
from scipy.integrate import trapezoid

def get_max_direction(result: DiagramResult):  
    data = pd.DataFrame({"phi": result.phi, "theta": result.theta, "D": result.D})
    phi_u = np.sort(data["phi"].unique())
    theta_u = np.sort(data["theta"].unique())
    pivot = data.pivot_table(index="theta", columns="phi", values="D", aggfunc='mean')

    pivot = pivot.reindex(index=theta_u, columns=phi_u)
    D_grid = pivot.values

    PHI, THETA = np.meshgrid(phi_u, theta_u, indexing='xy')
    X = D_grid * np.sin(THETA)**2 * np.cos(PHI)
    Y = D_grid * np.sin(THETA)**2 * np.sin(PHI)
    Z = D_grid * np.cos(THETA)*np.sin(THETA)
    
    Px_phi = np.trapezoid(X, phi_u, axis=1)
    Py_phi = np.trapezoid(Y, phi_u, axis=1)
    Pz_phi = np.trapezoid(Z, phi_u, axis=1)

    Px = np.trapezoid(Px_phi, theta_u)
    Py = np.trapezoid(Py_phi, theta_u)
    Pz = np.trapezoid(Pz_phi, theta_u)
    
    phi_eff = np.arctan2(Py, Px)
    theta_eff = np.arctan2(np.sqrt(Px**2 + Py**2), Pz)
    
    return [Px, Py, Pz], [phi_eff, theta_eff]

In [46]:
mix_D = diagTot.D - diagReg.D - diagSPP.D

diagMix = DiagramResult(diagReg.phi, diagReg.theta, mix_D)

In [105]:
%matplotlib inline

import pandas as pd
from MieSppForce.simulation import DiagramResult

def make_sphere(ax, radius=1, color='k', alpha=0.1, zorder=1):
    u = np.linspace(0, 2 * np.pi, 100)
    v = np.linspace(0, np.pi, 100)
    x = radius * np.outer(np.cos(u), np.sin(v))
    y = radius * np.outer(np.sin(u), np.sin(v))
    z = radius * np.outer(np.ones(np.size(u)), np.cos(v))+radius
    ax.plot_surface(x, y, z, color=color, alpha=alpha, zorder = zorder)
    
def make_plane(ax, size=1, color='k', alpha=0.1):
    xx, yy = np.meshgrid(np.linspace(-size, size, 10), np.linspace(-size, size, 10))
    zz = np.zeros_like(xx)
    ax.plot_surface(xx, yy, zz, color=color, alpha=alpha, zorder = 0)

def plot_3D_pattern(result: DiagramResult, ax, alpha=0.7, zorder=5, cmap = 'hot', color = None, label=None):
    data = pd.DataFrame({"phi": result.phi, "theta": result.theta, "D": result.D})
    phi_u = np.sort(data["phi"].unique())
    theta_u = np.sort(data["theta"].unique())
    pivot = data.pivot_table(index="theta", columns="phi", values="D", aggfunc='mean')

    pivot = pivot.reindex(index=theta_u, columns=phi_u)
    D_grid = pivot.values
    #D_grid = D_grid/max_val
    D_grid = D_grid / np.nanmax(D_grid)

    PHI, THETA = np.meshgrid(phi_u, theta_u, indexing='xy')
    X = D_grid * np.sin(THETA) * np.cos(PHI)
    Y = D_grid * np.sin(THETA) * np.sin(PHI)
    Z = D_grid * np.cos(THETA)
    
    norm = (D_grid - np.nanmin(D_grid)) / (np.nanmax(D_grid) - np.nanmin(D_grid) + 1e-20)
    
    cm = plt.get_cmap(cmap)
    colors = cm(norm)
    if color is not None:
        ax.plot_surface(X, Y, Z, color=color, rstride=1, cstride=1, linewidth=0, antialiased=True, alpha=alpha, zorder = zorder, label=label)
    else:
        ax.plot_surface(X, Y, Z, facecolors=colors, rstride=1, cstride=1, linewidth=0, antialiased=False, alpha=alpha, zorder = zorder, label=label)
    
    
    
max_val = np.nanmax(diagTot.D)



# Plot
fig = plt.figure(dpi=300)
fig.patch.set_alpha(0)
ax = fig.add_subplot(111, projection='3d',computed_zorder=False)

make_plane(ax, size=1, color='gold', alpha=0.6)

plot_3D_pattern(diagTot, ax, 1, 5, cmap='plasma', label='Air')
# plot_3D_pattern(diagSPP, ax, 1, 4, cmap='winter', label='SPP')

# ax.plot([0, np.sin(theta_eff)*np.cos(phi_eff)],
#         [0, np.sin(theta_eff)*np.sin(phi_eff)],
#         [0, np.cos(theta_eff)], color='black', linewidth=2, zorder=4, ls='--')

# sizer = 1.4

# phi_eff, theta_eff = get_max_direction(diagReg)[1]

# ax.quiver(
#     0, 0, 0,
#     np.sin(theta_eff)*np.cos(phi_eff)*sizer,
#     np.sin(theta_eff)*np.sin(phi_eff)*sizer,
#     np.cos(theta_eff)*sizer,
#     color='black', linewidth=2, zorder=4, arrow_length_ratio=0.15)

# plot_3D_pattern(diagReg, ax, 1, 4, cmap='hot', label='Air')

# make_sphere(ax, radius=0.1, color='red', alpha=1, zorder=10)



ax.set_box_aspect([1,1,1])

lim_ax = 1

ax.axes.set_xlim3d(left=-lim_ax, right=lim_ax) 
ax.axes.set_ylim3d(bottom=-lim_ax, top=lim_ax) 
ax.axes.set_zlim3d(bottom=0, top=1) 
ax.set_aspect('equal')

ax.grid(False)

ax.xaxis.pane.set_visible(False)  # Hide YZ plane
ax.yaxis.pane.set_visible(False)  # Hide XZ plane
ax.zaxis.line.set_visible(False)  # Hide Z axis line

ax.view_init(elev=10, azim=-25) 
# plt.legend()

# ax.set_xlabel('X')
# ax.set_ylabel('Y')
# ax.set_zlabel('Z')

ax.set_xticks(np.linspace(-lim_ax, lim_ax, 0))
ax.set_yticks(np.linspace(-lim_ax, lim_ax, 0))
ax.set_zticks([])

# ax.axis('off')

plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.savefig('3D_pattern.png', dpi=300, transparent=True)
plt.close()